<a href="https://colab.research.google.com/github/JosseAguirre/Taller-Practico-3---GRUPO-5---UIDE/blob/dev/Taller_Pr%C3%A1ctico_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from IPython.display import display

# 1. ARQUITECTURA Y GESTIÓN DE RUTAS
def configurar_entorno():
    """Crea la estructura de directorios estandarizada para el proyecto."""
    carpetas = ['data/raw', 'data/processed',]
    for carpeta in carpetas:
        Path(carpeta).mkdir(parents=True, exist_ok=True)
    print("✅ Estructura de directorios inicializada correctamente.")

# Ejecutar configuración
configurar_entorno()

✅ Estructura de directorios inicializada correctamente.


In [17]:
# 2. LECTURA E INTEGRACIÓN ESTRUCTURAL (CONCAT)

def unificar_entregas(ruta_matriz, ruta_ext) -> pd.DataFrame:
    """Lee y apila los archivos de entregas de ambas sedes desde la carpeta raw."""
    print("--- Iniciando Integración Estructural ---")

    # Lectura controlada (forzando IDs a string para no perder ceros a la izquierda)
    dtypes_dict = {'id_entrega': str, 'id_estudiante': str}

    # Se leen los archivos directamente usando los objetos Path enviados
    df_matriz = pd.read_csv(ruta_matriz, dtype=dtypes_dict, encoding='utf-8')
    df_ext = pd.read_csv(ruta_ext, dtype=dtypes_dict, encoding='utf-8')

    # Trazabilidad: Añadir origen antes de mezclar
    df_matriz['campus_origen_entrega'] = 'Matriz'
    df_ext['campus_origen_entrega'] = 'Extensión'

    # Integración estructural (Concatenación vertical)
    df_unificado = pd.concat([df_matriz, df_ext], ignore_index=True)

    # Control de calidad preventivo
    df_unificado = df_unificado.drop_duplicates(subset=['id_entrega'])

    print(f"✅ Archivos unificados desde data/raw/. Total de entregas: {len(df_unificado)}")
    return df_unificado

# Ejecución apuntando a la carpeta correcta
ruta_matriz_raw = Path('data/raw/entregas_campus_matriz.csv')
ruta_ext_raw = Path('data/raw/entregas_campus_extension.csv')

df_entregas_total = unificar_entregas(ruta_matriz_raw, ruta_ext_raw)
display(df_entregas_total.head(3))

--- Iniciando Integración Estructural ---
✅ Archivos unificados desde data/raw/. Total de entregas: 300


,id_entrega,id_estudiante,fecha_subida,materia,tipo_proyecto,puntaje_obtenido,estado_entrega,campus_origen_entrega
0,PRJ-1001,E001,2026-08-01,Sistemas de Bases de Datos,Modelado ER,95.5,Aprobado,Matriz
1,PRJ-1002,E002,2026-08-02,Desarrollo de Software,Prototipo,88.0,Aprobado,Matriz
2,PRJ-1003,E003,2026-08-03,DataOps,Pipeline,92.0,Aprobado,Matriz


In [18]:
# 3. LECTURA DE EXCEL, INTEGRACIÓN RELACIONAL Y AUDITORÍA
def cruzar_con_maestro(df_transaccional: pd.DataFrame, ruta_maestro) -> pd.DataFrame:
    print("--- Iniciando Integración Relacional y Auditoría ---")

    # Lectura del archivo Excel Maestro desde la carpeta raw
    df_master = pd.read_excel(ruta_maestro, dtype={'id_estudiante': str})

    # Cruce (Left Join) con validaciones de calidad
    df_merged = df_transaccional.merge(
        df_master,
        on='id_estudiante',
        how='left',
        validate='m:1',
        indicator=True
    )

    # Extracción de Métricas de Calidad
    total_procesados = len(df_merged)
    huerfanos = len(df_merged[df_merged['_merge'] == 'left_only'])
    nulos_puntaje = df_merged['puntaje_obtenido'].isna().sum()

    print("\n📊 REPORTE DE INDICADORES DE CALIDAD:")
    print(f" 🔹 Registros totales procesados: {total_procesados}")
    print(f" 🔹 Claves sin correspondencia (Estudiantes Huérfanos): {huerfanos}")
    print(f" 🔹 Valores nulos críticos detectados (Entregas sin puntaje): {nulos_puntaje}")

    return df_merged

# Ejecución apuntando a la carpeta correcta
ruta_maestro_raw = Path('data/raw/estudiantes_master.xlsx')

df_consolidado = cruzar_con_maestro(df_entregas_total, ruta_maestro_raw)

--- Iniciando Integración Relacional y Auditoría ---

📊 REPORTE DE INDICADORES DE CALIDAD:
 🔹 Registros totales procesados: 300
 🔹 Claves sin correspondencia (Estudiantes Huérfanos): 5
 🔹 Valores nulos críticos detectados (Entregas sin puntaje): 9


In [19]:
# 4. LIMPIEZA DE DATOS Y TRATAMIENTO
def aplicar_limpieza(df: pd.DataFrame) -> pd.DataFrame:
    print("\n--- Aplicando Reglas de Limpieza ---")
    df_clean = df.copy()

    # Tratamiento de Fechas (Controlando nulos post-coerción)
    for col_fecha in ['fecha_subida', 'fecha_nacimiento']:
        if col_fecha in df_clean.columns:
            nulos_antes = df_clean[col_fecha].isna().sum()
            df_clean[col_fecha] = pd.to_datetime(df_clean[col_fecha], errors='coerce')
            nulos_despues = df_clean[col_fecha].isna().sum()
            invalidados = nulos_despues - nulos_antes
            print(f" -> Columna '{col_fecha}': {invalidados} fechas invalidadas tras coerción.")

    # Tratamiento de Estudiantes Huérfanos (Imputación por defecto)
    columnas_maestro = ['nombre_completo', 'carrera', 'campus_origen', 'tipo_beca', 'estado_academico']
    for col in columnas_maestro:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna("No Registrado en Maestro")

    # Eliminar columna de auditoría interna
    if '_merge' in df_clean.columns:
        df_clean = df_clean.drop(columns=['_merge'])

    print("✅ Limpieza y estandarización finalizada.")
    return df_clean

# Ejecución
df_final = aplicar_limpieza(df_consolidado)
display(df_final.head(3))


--- Aplicando Reglas de Limpieza ---
 -> Columna 'fecha_subida': 0 fechas invalidadas tras coerción.
 -> Columna 'fecha_nacimiento': 0 fechas invalidadas tras coerción.
✅ Limpieza y estandarización finalizada.


,id_entrega,id_estudiante,fecha_subida,materia,tipo_proyecto,puntaje_obtenido,estado_entrega,campus_origen_entrega,nombre_completo,fecha_nacimiento,carrera,campus_origen,tipo_beca,estado_academico
0,PRJ-1001,E001,2026-08-01,Sistemas de Bases de Datos,Modelado ER,95.5,Aprobado,Matriz,Mateo Alvarado,1998-05-12,Ingeniería de Software,Matriz,Completa,Activo
1,PRJ-1002,E002,2026-08-02,Desarrollo de Software,Prototipo,88.0,Aprobado,Matriz,Camila Sánchez,2000-02-20,Ingeniería de Software,Matriz,Parcial,Activo
2,PRJ-1003,E003,2026-08-03,DataOps,Pipeline,92.0,Aprobado,Matriz,Santiago Ráos,1999-03-10,Ciencias de Datos,Matriz,Ninguna,Activo
